# RL Trading Agent — 4h Timeframe Experiment

**4h BTC/USDT**: embeddings (95 features) vs baseline (18 features), linear LR, [512,256]/[256,256] network, 2M steps x 7 seeds

## Инструкция

### Google Colab:
1. Runtime -> Change runtime type -> GPU (T4)
2. Загрузи файлы через Files panel (слева):
   - `btc_4h_embedding_features.parquet`
   - `btc_4h_features.parquet`
3. Измени `BASE_DIR` в ячейке 2 на `/content`

### DataSphere:
1. Выбери конфигурацию с GPU (g1.1 = V100)
2. Перетащи два файла в `/home/jupyter/project/`
3. Запускай ячейки по порядку

**Гипотеза:** на 4h таймфрейме NLP-фичи полезнее, потому что новости ещё не успели отразиться в цене.

**Предварительные результаты (CPU, 1M steps, 3 seeds):**
| Agent | Sharpe | Return | MaxDD |
|-------|--------|--------|-------|
| Embeddings 4h | 0.528 +/- 0.64 | 15.8% | 10.8% |
| Baseline 4h | 0.360 +/- 0.26 | 20.8% | 19.7% |

## 1. Установка зависимостей

In [ ]:
%pip install -q stable-baselines3>=2.1.0 gymnasium>=0.29.0 pandas pyarrow matplotlib

## 2. Пути к файлам

In [ ]:
import os

BASE_DIR = '/home/jupyter/project'
DATA_DIR      = f'{BASE_DIR}'
RESULTS_DIR   = f'{BASE_DIR}/results'
MODELS_DIR    = f'{BASE_DIR}/experiments'

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

for fname in ['btc_4h_embedding_features.parquet', 'btc_4h_features.parquet']:
    path = f'{DATA_DIR}/{fname}'
    exists = os.path.exists(path)
    size_mb = os.path.getsize(path) / 1e6 if exists else 0
    print(f"{'✅' if exists else '❌  ЗАГРУЗИ ФАЙЛ!'} {fname}: {size_mb:.1f} MB")

## 3. Исходный код

In [ ]:
# ── TradingEnv ───────────────────────────────────────────────────────────────
import gymnasium as gym
import numpy as np


class TradingEnv(gym.Env):
    metadata = {'render_modes': []}

    def __init__(self, features, prices, window=30, tx_cost=0.001,
                 reward_type='basic', allow_short=False, volatility_penalty=0.5):
        super().__init__()
        assert len(features) == len(prices)
        assert len(features) > window
        self.features = features.astype(np.float32)
        self.prices = prices.astype(np.float64)
        self.window = window
        self.tx_cost = tx_cost
        self.reward_type = reward_type
        self.allow_short = allow_short
        self.volatility_penalty = volatility_penalty
        n_features = features.shape[1]
        self.observation_space = gym.spaces.Box(
            low=-np.inf, high=np.inf, shape=(window * n_features,), dtype=np.float32)
        action_low = -1.0 if allow_short else 0.0
        self.action_space = gym.spaces.Box(
            low=action_low, high=1.0, shape=(1,), dtype=np.float32)
        self.current_step = 0
        self.prev_allocation = 0.0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = self.window
        self.prev_allocation = 0.0
        return self._get_obs(), {}

    def step(self, action):
        allocation = float(np.clip(action[0], self.action_space.low[0], 1.0))
        price_current = self.prices[self.current_step - 1]
        price_next    = self.prices[self.current_step]
        log_return    = float(np.log(price_next / price_current))
        delta         = abs(allocation - self.prev_allocation)
        tx_penalty    = self.tx_cost * delta
        if self.reward_type == 'risk_adjusted':
            reward = float(log_return * allocation - tx_penalty - self.volatility_penalty * delta)
        else:
            reward = float(log_return * allocation - tx_penalty)
        self.prev_allocation = allocation
        self.current_step += 1
        terminated = self.current_step >= len(self.prices) - 1
        info = {'log_return': log_return, 'allocation': allocation}
        return self._get_obs(), reward, terminated, False, info

    def _get_obs(self):
        start = self.current_step - self.window
        return self.features[start:self.current_step].flatten()


print('✅ TradingEnv loaded')

In [ ]:
# ── Метрики ───────────────────────────────────────────────────────────────────
import numpy as np

TRADING_DAYS_PER_YEAR = 365


def compute_metrics(returns):
    returns   = np.asarray(returns, dtype=np.float64)
    total_ret = float(np.prod(1 + returns) - 1)
    mean_r    = np.mean(returns)
    std_r     = np.std(returns, ddof=1) if len(returns) > 1 else 1.0
    sharpe    = float((mean_r / std_r) * np.sqrt(TRADING_DAYS_PER_YEAR)) if std_r > 0 else 0.0
    downside  = returns[returns < 0]
    down_std  = float(np.std(downside, ddof=1)) if len(downside) > 1 else 1.0
    sortino   = float((mean_r / down_std) * np.sqrt(TRADING_DAYS_PER_YEAR)) if down_std > 0 else 0.0
    cum       = np.cumprod(1 + returns)
    max_dd    = float(np.max(1 - cum / np.maximum.accumulate(cum)))
    ann_ret   = (1 + total_ret) ** (TRADING_DAYS_PER_YEAR / max(len(returns), 1)) - 1
    calmar    = float(ann_ret / max_dd) if max_dd > 0 else 0.0
    return dict(total_return=total_ret, sharpe_ratio=sharpe, sortino_ratio=sortino,
                max_drawdown=max_dd, calmar_ratio=calmar)


print('✅ compute_metrics loaded')

In [ ]:
# ── Backtest ──────────────────────────────────────────────────────────────────
import torch
import numpy as np
from stable_baselines3 import PPO, A2C, SAC


def load_model(model_path):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    for cls in [PPO, A2C, SAC]:
        try:
            return cls.load(model_path, device=device)
        except Exception:
            continue
    raise ValueError(f'Cannot load model: {model_path}')


def run_backtest(features, prices, model_path, window=30, tx_cost=0.001):
    env   = TradingEnv(features=features, prices=prices, window=window, tx_cost=tx_cost)
    model = load_model(model_path)
    obs, _ = env.reset()
    daily_returns, allocations = [], []
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, _, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        daily_returns.append(np.exp(info['log_return'] * info['allocation']) - 1)
        allocations.append(info['allocation'])
    daily_returns = np.array(daily_returns)
    equity_curve  = np.cumprod(1 + daily_returns)
    return dict(metrics=compute_metrics(daily_returns),
                daily_returns=daily_returns,
                allocations=np.array(allocations),
                equity_curve=equity_curve)


print('✅ run_backtest loaded')

In [ ]:
# ── AgentConfig + train_agent ─────────────────────────────────────────────────
import torch
import logging
from dataclasses import dataclass, field
from typing import List
from datetime import datetime
from pathlib import Path
from stable_baselines3 import PPO, A2C, SAC
from stable_baselines3.common.callbacks import BaseCallback

logging.basicConfig(level=logging.WARNING)


@dataclass
class AgentConfig:
    learning_rate: float = 3e-4
    n_steps: int = 2048
    batch_size: int = 64
    n_epochs: int = 10
    gamma: float = 0.99
    gae_lambda: float = 0.95
    clip_range: float = 0.2
    net_arch: List[int] = field(default_factory=lambda: [256, 256])
    activation_fn: str = 'tanh'
    lr_schedule: str = 'constant'
    total_timesteps: int = 500_000
    seed: int = 42
    algorithm: str = 'PPO'
    window: int = 30
    tx_cost: float = 0.001
    reward_type: str = 'basic'
    allow_short: bool = False
    agent_type: str = 'baseline'
    save_dir: str = MODELS_DIR

    def policy_kwargs(self):
        return {'net_arch': self.net_arch,
                'activation_fn': {'tanh': torch.nn.Tanh, 'relu': torch.nn.ReLU}[self.activation_fn]}


EMBEDDINGS_NET_ARCH = [512, 256]


def _linear_schedule(initial_lr):
    return lambda progress_remaining: progress_remaining * initial_lr


class ProgressCallback(BaseCallback):
    def __init__(self, total_timesteps, print_every=100_000):
        super().__init__()
        self.total_timesteps = total_timesteps
        self.print_every = print_every
        self._last_print = 0

    def _on_step(self):
        if self.num_timesteps - self._last_print >= self.print_every:
            pct = 100 * self.num_timesteps / self.total_timesteps
            print(f'  [{pct:5.1f}%] {self.num_timesteps:>8}/{self.total_timesteps}', flush=True)
            self._last_print = self.num_timesteps
        return True


def train_agent(config, features, prices):
    device   = 'cuda' if torch.cuda.is_available() else 'cpu'
    algo_cls = {'PPO': PPO, 'A2C': A2C, 'SAC': SAC}[config.algorithm]

    lr = (_linear_schedule(config.learning_rate)
          if config.lr_schedule == 'linear' else config.learning_rate)

    policy_kwargs = config.policy_kwargs()
    if config.agent_type in ('embeddings', 'fusion') and config.net_arch == [256, 256]:
        policy_kwargs['net_arch'] = EMBEDDINGS_NET_ARCH

    env = TradingEnv(features=features, prices=prices, window=config.window,
                     tx_cost=config.tx_cost, reward_type=config.reward_type,
                     allow_short=config.allow_short)

    kwargs = dict(learning_rate=lr, seed=config.seed, verbose=0,
                  device=device, policy_kwargs=policy_kwargs)
    if config.algorithm in ('PPO', 'A2C'):
        kwargs.update(n_steps=config.n_steps, gamma=config.gamma, gae_lambda=config.gae_lambda)
    if config.algorithm == 'PPO':
        kwargs.update(batch_size=config.batch_size, n_epochs=config.n_epochs,
                      clip_range=config.clip_range)
    if config.algorithm == 'SAC':
        kwargs.update(gamma=config.gamma, batch_size=config.batch_size)

    model = algo_cls('MlpPolicy', env, **kwargs)
    model.learn(total_timesteps=config.total_timesteps,
                callback=ProgressCallback(config.total_timesteps))

    save_dir = Path(config.save_dir) / config.agent_type / datetime.now().strftime('%Y%m%d_%H%M%S')
    save_dir.mkdir(parents=True, exist_ok=True)
    model_path = save_dir / 'model.zip'
    model.save(str(model_path.with_suffix('')))
    return model_path


print(f'✅ train_agent loaded | device: {"cuda" if torch.cuda.is_available() else "cpu"}')

## 4. Загрузка данных (4h)

In [ ]:
import pandas as pd
import numpy as np

_PRICE_COLS = {'open', 'high', 'low', 'close', 'volume', 'raw_close'}


def load_parquet(path, date_start, date_end):
    df = pd.read_parquet(path)
    start_ts = pd.Timestamp(date_start, tz='UTC')
    end_ts   = pd.Timestamp(date_end,   tz='UTC')
    df = df.loc[start_ts:end_ts]
    prices   = df['raw_close'].to_numpy(dtype=np.float64) if 'raw_close' in df.columns \
               else df['close'].to_numpy(dtype=np.float64)
    feat_cols = [c for c in df.columns if c.lower() not in _PRICE_COLS]
    features  = np.nan_to_num(df[feat_cols].to_numpy(dtype=np.float32))
    print(f'  {path.split("/")[-1]}: {len(df)} rows, {features.shape[1]} features')
    return features, prices


TRAIN_START, TRAIN_END = '2020-01-01', '2023-12-31'
TEST_START,  TEST_END  = '2024-01-01', '2024-12-31'

print('Train data (4h):')
emb_train_feat,  emb_train_prices  = load_parquet(f'{DATA_DIR}/btc_4h_embedding_features.parquet', TRAIN_START, TRAIN_END)
base_train_feat, base_train_prices = load_parquet(f'{DATA_DIR}/btc_4h_features.parquet',           TRAIN_START, TRAIN_END)

print('Test data (4h):')
emb_test_feat,   emb_test_prices   = load_parquet(f'{DATA_DIR}/btc_4h_embedding_features.parquet', TEST_START, TEST_END)
base_test_feat,  base_test_prices  = load_parquet(f'{DATA_DIR}/btc_4h_features.parquet',           TEST_START, TEST_END)

print('✅ 4h data loaded')

## 5. Обучение Embeddings (4h) — 7 seeds × 2M шагов

In [ ]:
import time, csv

SEEDS = [42, 43, 44, 45, 46, 47, 48]
TOTAL_TIMESTEPS = 2_000_000

emb_results = []

for i, seed in enumerate(SEEDS, 1):
    print(f'\n{"="*55}')
    print(f'  4h Embeddings  seed={seed}  ({i}/{len(SEEDS)})')
    print(f'{"="*55}')
    t0 = time.time()

    config = AgentConfig(
        agent_type='embeddings', algorithm='PPO',
        total_timesteps=TOTAL_TIMESTEPS, seed=seed,
        lr_schedule='linear', save_dir=MODELS_DIR,
    )
    model_path = train_agent(config, emb_train_feat, emb_train_prices)
    bt = run_backtest(emb_test_feat, emb_test_prices, str(model_path))
    m  = bt['metrics']
    emb_results.append(dict(agent_type='embeddings_4h', asset='BTC/USDT', seed=seed, **m))

    print(f'  ✅ Sharpe={m["sharpe_ratio"]:.3f}  Return={m["total_return"]*100:.1f}%  '
          f'MaxDD={m["max_drawdown"]*100:.1f}%  [{(time.time()-t0)/60:.1f} min]')

print('\n✅ 4h Embeddings training complete')

## 6. Обучение Baseline (4h) — 7 seeds × 2M шагов

In [ ]:
base_results = []

for i, seed in enumerate(SEEDS, 1):
    print(f'\n{"="*55}')
    print(f'  4h Baseline  seed={seed}  ({i}/{len(SEEDS)})')
    print(f'{"="*55}')
    t0 = time.time()

    config = AgentConfig(
        agent_type='baseline', algorithm='PPO',
        total_timesteps=TOTAL_TIMESTEPS, seed=seed,
        lr_schedule='linear', save_dir=MODELS_DIR,
    )
    model_path = train_agent(config, base_train_feat, base_train_prices)
    bt = run_backtest(base_test_feat, base_test_prices, str(model_path))
    m  = bt['metrics']
    base_results.append(dict(agent_type='baseline_4h', asset='BTC/USDT', seed=seed, **m))

    print(f'  ✅ Sharpe={m["sharpe_ratio"]:.3f}  Return={m["total_return"]*100:.1f}%  '
          f'MaxDD={m["max_drawdown"]*100:.1f}%  [{(time.time()-t0)/60:.1f} min]')

print('\n✅ 4h Baseline training complete')

## 7. Результаты: 4h embeddings vs 4h baseline vs daily

In [ ]:
import pandas as pd
import numpy as np

all_results = emb_results + base_results
df = pd.DataFrame(all_results)

print('\n' + '='*72)
print(f'{"Agent":<18} {"Sharpe":>14} {"Return":>13} {"MaxDD":>13} {"Sortino":>12}')
print('='*72)
for agent in ['embeddings_4h', 'baseline_4h']:
    r = df[df.agent_type == agent]
    print(f'{agent:<18}  '
          f'{r.sharpe_ratio.mean():>5.3f} ± {r.sharpe_ratio.std():.3f}  '
          f'{r.total_return.mean()*100:>5.1f}% ± {r.total_return.std()*100:.1f}%  '
          f'{r.max_drawdown.mean()*100:>4.1f}% ± {r.max_drawdown.std()*100:.1f}%  '
          f'{r.sortino_ratio.mean():>5.3f} ± {r.sortino_ratio.std():.3f}')
print('='*72)

print('\n--- Daily results (для сравнения) ---')
print('  Embeddings daily: Sharpe=X.XXX  Return=XX.X%  MaxDD=XX.X%')
print('  Baseline daily:   Sharpe=X.XXX  Return=XX.X%  MaxDD=XX.X%')
print('  (подставь реальные результаты из colab_training.ipynb)')

e4h = df[df.agent_type == 'embeddings_4h']
b4h = df[df.agent_type == 'baseline_4h']
delta_sharpe = e4h.sharpe_ratio.mean() - b4h.sharpe_ratio.mean()
print(f'\n  4h: Embeddings - Baseline = {delta_sharpe:+.3f} Sharpe')
print(f'  Гипотеза (NLP полезнее на 4h): {"✅ ПОДТВЕРЖДЕНА" if delta_sharpe > 0 else "❌ НЕ ПОДТВЕРЖДЕНА"}')

## 8. Сохранение результатов

In [ ]:
import csv

CSV_FIELDS = ['agent_type', 'asset', 'seed',
              'total_return', 'sharpe_ratio', 'sortino_ratio', 'max_drawdown', 'calmar_ratio']

results_path = f'{RESULTS_DIR}/4h_experiment_results.csv'
with open(results_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)
    writer.writeheader()
    writer.writerows(all_results)

print(f'✅ CSV: {results_path}')
print(df[CSV_FIELDS].to_string(index=False))

## 9. График Sharpe по seeds (4h)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (label, results) in zip(axes, [
    ('4h Embeddings (95 feat, [512,256], 2M)', emb_results),
    ('4h Baseline (18 feat, [256,256], 2M)',   base_results),
]):
    sharpes = [r['sharpe_ratio'] for r in results]
    ax.bar(range(len(SEEDS)), sharpes, color='steelblue', alpha=0.75, edgecolor='white')
    ax.axhline(np.mean(sharpes), color='red', linestyle='--',
               label=f'mean = {np.mean(sharpes):.3f} ± {np.std(sharpes):.3f}')
    ax.set_xticks(range(len(SEEDS)))
    ax.set_xticklabels([f'seed {s}' for s in SEEDS])
    ax.set_ylabel('Sharpe Ratio')
    ax.set_title(label)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = f'{RESULTS_DIR}/4h_sharpe_comparison.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Plot: {fig_path}')

## 10. Equity Curves

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Re-run backtests to get equity curves
fig, ax = plt.subplots(1, 1, figsize=(14, 6))

# Find best seed for each agent type
best_emb = max(emb_results, key=lambda r: r['sharpe_ratio'])
best_base = max(base_results, key=lambda r: r['sharpe_ratio'])

# Re-backtest best models to get curves
from pathlib import Path
import glob

# Plot all equity curves
colors_emb = plt.cm.Blues(np.linspace(0.4, 0.9, len(SEEDS)))
colors_base = plt.cm.Oranges(np.linspace(0.4, 0.9, len(SEEDS)))

for i, (emb_r, base_r) in enumerate(zip(emb_results, base_results)):
    seed = SEEDS[i]
    # Find model paths
    emb_dirs = sorted(glob.glob(f'{MODELS_DIR}/embeddings/*/model.zip'))
    base_dirs = sorted(glob.glob(f'{MODELS_DIR}/baseline/*/model.zip'))

    if i < len(emb_dirs):
        bt_emb = run_backtest(emb_test_feat, emb_test_prices, emb_dirs[i].replace('.zip', ''))
        ax.plot(bt_emb['equity_curve'], color=colors_emb[i], alpha=0.6,
                label=f'Emb s{seed} (Sharpe={emb_r["sharpe_ratio"]:.2f})' if i == 0 else f'Emb s{seed} ({emb_r["sharpe_ratio"]:.2f})')

    if i < len(base_dirs):
        bt_base = run_backtest(base_test_feat, base_test_prices, base_dirs[i].replace('.zip', ''))
        ax.plot(bt_base['equity_curve'], color=colors_base[i], alpha=0.6, linestyle='--',
                label=f'Base s{seed} ({base_r["sharpe_ratio"]:.2f})')

ax.axhline(1.0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('4h candles (2024)')
ax.set_ylabel('Portfolio Value')
ax.set_title('4h Equity Curves: Embeddings vs Baseline (all seeds)')
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = f'{RESULTS_DIR}/4h_equity_curves.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Plot: {fig_path}')

## 11. (Optional) SAC Comparison

SAC (Soft Actor-Critic) is more stable than PPO but slower to train.
On GPU this should take ~30 min per seed. Skip this section if short on time.

In [ ]:
SAC_SEEDS = [42, 43, 44]
SAC_TIMESTEPS = 1_000_000

sac_emb_results = []
sac_base_results = []

# SAC Embeddings
for i, seed in enumerate(SAC_SEEDS, 1):
    print(f'\n{"="*55}')
    print(f'  4h SAC Embeddings  seed={seed}  ({i}/{len(SAC_SEEDS)})')
    print(f'{"="*55}')
    t0 = time.time()

    config = AgentConfig(
        agent_type='embeddings', algorithm='SAC',
        total_timesteps=SAC_TIMESTEPS, seed=seed,
        save_dir=MODELS_DIR,
    )
    model_path = train_agent(config, emb_train_feat, emb_train_prices)
    bt = run_backtest(emb_test_feat, emb_test_prices, str(model_path))
    m  = bt['metrics']
    sac_emb_results.append(dict(agent_type='sac_embeddings_4h', asset='BTC/USDT', seed=seed, **m))

    print(f'  Sharpe={m["sharpe_ratio"]:.3f}  Return={m["total_return"]*100:.1f}%  '
          f'MaxDD={m["max_drawdown"]*100:.1f}%  [{(time.time()-t0)/60:.1f} min]')

# SAC Baseline
for i, seed in enumerate(SAC_SEEDS, 1):
    print(f'\n{"="*55}')
    print(f'  4h SAC Baseline  seed={seed}  ({i}/{len(SAC_SEEDS)})')
    print(f'{"="*55}')
    t0 = time.time()

    config = AgentConfig(
        agent_type='baseline', algorithm='SAC',
        total_timesteps=SAC_TIMESTEPS, seed=seed,
        save_dir=MODELS_DIR,
    )
    model_path = train_agent(config, base_train_feat, base_train_prices)
    bt = run_backtest(base_test_feat, base_test_prices, str(model_path))
    m  = bt['metrics']
    sac_base_results.append(dict(agent_type='sac_baseline_4h', asset='BTC/USDT', seed=seed, **m))

    print(f'  Sharpe={m["sharpe_ratio"]:.3f}  Return={m["total_return"]*100:.1f}%  '
          f'MaxDD={m["max_drawdown"]*100:.1f}%  [{(time.time()-t0)/60:.1f} min]')

# Summary
sac_df = pd.DataFrame(sac_emb_results + sac_base_results)
print('\n' + '='*72)
print('SAC Results:')
print('='*72)
for agent in ['sac_embeddings_4h', 'sac_baseline_4h']:
    r = sac_df[sac_df.agent_type == agent]
    print(f'{agent:<22}  '
          f'Sharpe={r.sharpe_ratio.mean():.3f} +/- {r.sharpe_ratio.std():.3f}  '
          f'Return={r.total_return.mean()*100:.1f}%  '
          f'MaxDD={r.max_drawdown.mean()*100:.1f}%')
print('='*72)

# Save SAC results
sac_path = f'{RESULTS_DIR}/4h_sac_results.csv'
with open(sac_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)
    writer.writeheader()
    writer.writerows(sac_emb_results + sac_base_results)
print(f'CSV: {sac_path}')